# ★ Додаткове завдання: функція `clean_dataset(df)`
**Виконав:** Мельниківський Андрій, група КН-41

In [1]:
import pandas as pd
import numpy as np

### Функція
Кроки очищення (у зазначеному порядку):
1. видалення повних дублікатів;
2. заповнення числових пропусків медіаною, категоріальних — модою;
3. перетворення колонок з датою на `datetime64` (колонки, назва яких містить `date`/`time`, або вказані в `date_cols`);
4. видалення рядків із викидами за IQR по **всіх** числових колонках.

Функція повертає **копію**, оригінал не змінюється.

In [2]:
def clean_dataset(df, date_cols=None):
    """Повертає очищену копію DataFrame."""
    out = df.copy()
    # визначаємо колонки з датою
    if date_cols is None:
        date_cols = [c for c in out.columns if any(k in str(c).lower() for k in ('date', 'time'))]

    # 1) повні дублікати
    out = out.drop_duplicates()

    # 2) пропуски: числові -> медіана, категоріальні -> мода
    for col in out.columns:
        if col in date_cols:
            continue
        if pd.api.types.is_numeric_dtype(out[col]):
            out[col] = out[col].fillna(out[col].median())
        else:
            mode = out[col].mode()
            if not mode.empty:
                out[col] = out[col].fillna(mode[0])

    # 3) дати -> datetime64
    for col in date_cols:
        out[col] = pd.to_datetime(out[col], errors='coerce')

    # 4) викиди за IQR для всіх числових колонок
    num_cols = out.select_dtypes(include='number').columns
    mask = pd.Series(True, index=out.index)
    for col in num_cols:
        Q1, Q3 = out[col].quantile(0.25), out[col].quantile(0.75)
        IQR = Q3 - Q1
        mask &= out[col].between(Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)
    return out[mask].reset_index(drop=True)

### Тест на штучно створеному «брудному» датасеті

In [3]:
dirty = pd.DataFrame({
    'order_date': ['2025-03-01', '2025-03-02', '2025-03-02', '2025-03-03', '2025-03-04', '2025-03-04',
                   '2025-03-05', 'not a date', '2025-03-06', '2025-03-07', '2025-03-08', '2025-03-09'],
    'product':  ['Latte', 'Americano', 'Americano', 'Cocoa', None, None, 'Latte', 'Espresso', 'Latte', 'Cocoa', 'Latte', 'Americano'],
    'category': ['hot', 'hot', 'hot', 'sweet', 'hot', 'hot', 'hot', 'hot', None, 'sweet', 'hot', 'hot'],
    'price':    [38.7, 28.9, 28.9, 38.7, 28.9, 28.9, None, 24.0, 38.7, 38.7, 999.0, 28.9],
    'quantity': [1, 2, 2, 1, 3, 3, 1, None, 2, 1, 1, 150]
})
print('ДО очищення:', dirty.shape, '| пропусків:', dirty.isnull().sum().sum(), '| дублікатів:', dirty.duplicated().sum())
dirty

ДО очищення: (12, 5) | пропусків: 5 | дублікатів: 2


,order_date,product,category,price,quantity
0,2025-03-01,Latte,hot,38.7,1.0
1,2025-03-02,Americano,hot,28.9,2.0
2,2025-03-02,Americano,hot,28.9,2.0
3,2025-03-03,Cocoa,sweet,38.7,1.0
4,2025-03-04,None,hot,28.9,3.0
5,2025-03-04,None,hot,28.9,3.0
6,2025-03-05,Latte,hot,NaN,1.0
7,not a date,Espresso,hot,24.0,NaN
8,2025-03-06,Latte,None,38.7,2.0
9,2025-03-07,Cocoa,sweet,38.7,1.0


In [4]:
cleaned = clean_dataset(dirty)
print('ПІСЛЯ очищення:', cleaned.shape, '| пропусків:', cleaned.isnull().sum().sum(), '| дублікатів:', cleaned.duplicated().sum())
print(cleaned.dtypes)
cleaned

ПІСЛЯ очищення: (8, 5) | пропусків: 1 | дублікатів: 0
order_date    datetime64[ns]
product               object
category              object
price                float64
quantity             float64
dtype: object


,order_date,product,category,price,quantity
0,2025-03-01,Latte,hot,38.7,1.0
1,2025-03-02,Americano,hot,28.9,2.0
2,2025-03-03,Cocoa,sweet,38.7,1.0
3,2025-03-04,Latte,hot,28.9,3.0
4,2025-03-05,Latte,hot,38.7,1.0
5,NaT,Espresso,hot,24.0,1.0
6,2025-03-06,Latte,hot,38.7,2.0
7,2025-03-07,Cocoa,sweet,38.7,1.0


Видалено дублікати (2 рядки), заповнено пропуски (медіана/мода), `order_date` стала `datetime64` (некоректна дата → `NaT`), рядки з викидами `price = 999` та `quantity = 150` відфільтровано за IQR.

In [5]:
cleaned.to_csv('cleaned_dataset.csv', index=False)
print('Збережено: cleaned_dataset.csv')
print('Оригінал не змінено:', dirty.shape)

Збережено: cleaned_dataset.csv
Оригінал не змінено: (12, 5)
